# Sprawozdanie L5 — SAC + HER (PandaReach-v3)

**Autor:** (uzupełnij)  
**Cel:** implementacja *Hindsight Experience Replay* oraz automatycznego współczynnika entropii $\alpha$ w SAC; trening i analiza wyników.

## Uruchomienie notebooka

- **Zalecane:** uruchom serwer Jupyter / VS Code z **katalogiem roboczym `l5/`** (tam są `train.py`, `asdf/`, `runs/`, `weights/`).
- Pierwsza komórka kodu szuka katalogu zawierającego `train.py` i `asdf/` (w górę od `Path.cwd()`), ustawia `os.chdir` i dodaje go do `sys.path`.

**Ścieżki (domyślne):**

| Zmienna | Znaczenie |
|---------|-----------|
| `RUN_DIR` | katalog z `metadata.json` + `policy.pt` (checkpoint) |
| `RUNS_DIR` | katalog TensorBoard (`runs/`) |

Dla tego ćwiczenia domyślnie: `weights/2026-05-24_16-53` oraz najnowszy podkatalog w `runs/` (po czasie modyfikacji plików).

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_l5_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "train.py").is_file() and (p / "asdf").is_dir():
            return p
    raise FileNotFoundError(
        "Nie znaleziono katalogu l5 (train.py + asdf/). "
        "Ustaw cwd na UMISI-sem1-RL/l5/ i uruchom ponownie."
    )


L5_ROOT = find_l5_root()
os.chdir(L5_ROOT)
if str(L5_ROOT) not in sys.path:
    sys.path.insert(0, str(L5_ROOT))

# Checkpoint treningu (zmień jeśli masz inny run)
RUN_DIR = L5_ROOT / "weights" / "2026-05-24_16-53"
RUNS_DIR = L5_ROOT / "runs"

print("L5_ROOT =", L5_ROOT)
print("RUN_DIR =", RUN_DIR)
print("RUNS_DIR =", RUNS_DIR)

L5_ROOT = C:\Users\womackow\Zbiorczy\_WIN_AI_AGH_\WIN_sem_1\UMISI-sem1-RL\l5
RUN_DIR = C:\Users\womackow\Zbiorczy\_WIN_AI_AGH_\WIN_sem_1\UMISI-sem1-RL\l5\weights\2026-05-24_16-53
RUNS_DIR = C:\Users\womackow\Zbiorczy\_WIN_AI_AGH_\WIN_sem_1\UMISI-sem1-RL\l5\runs


## Hiperparametry z `metadata.json`

Poniżej pełna sygnatura zapisana przy treningu — musi być zgodna z aktualnym `train.py`, inaczej `python train.py --load ...` zgłosi błąd.

In [2]:
meta_path = RUN_DIR / "metadata.json"
if not meta_path.is_file():
    raise FileNotFoundError(meta_path)

with meta_path.open(encoding="utf-8") as f:
    meta_doc = json.load(f)

sig = meta_doc["signature"]
print(json.dumps(sig, indent=2, ensure_ascii=False))

{
  "env_id": "PandaReach-v3",
  "eval_render": {
    "n_episodes": 50,
    "sleep": 0.03333333333333333
  },
  "format_version": 1,
  "her": {
    "goal_selection_strategy": "future",
    "n_sampled_goal": 3,
    "replay_buffer_size_load": 50000,
    "replay_buffer_size_train": 1000000
  },
  "policy": {
    "extractor": "DictExtractor",
    "hidden_sizes": [
      64,
      64
    ]
  },
  "sac": {
    "alpha": "auto",
    "batch_size": 64,
    "gamma": 0.9,
    "lr": 0.0001,
    "max_episode_len": 100,
    "n_test_episodes": 25,
    "n_updates": null,
    "polyak": 0.995,
    "seed": 0,
    "start_steps": 1000,
    "target_entropy": "auto",
    "update_after": 1000,
    "update_every": 1
  },
  "train_loop": {
    "log_interval": 1000,
    "n_steps": 100000
  }
}


## Implementacja — HER

Pełny moduł: [`asdf/buffers.py`](asdf/buffers.py) (klasa `HerReplayBuffer`). Poniżej źródło kluczowych metod: episodyczne zbieranie przejść, na końcu epizodu zapis z oryginalnym celem oraz `n_sampled_goal` wariantów z nowym `desired_goal` i nagrodą z `env.unwrapped.compute_reward`, strategia wyboru celu (`future` / `final` / `episode`).

In [3]:
import inspect

from asdf.buffers import HerReplayBuffer

print("--- HerReplayBuffer.end_episode ---\n")
print(inspect.getsource(HerReplayBuffer.end_episode))
print("\n--- HerReplayBuffer._sample_goal ---\n")
print(inspect.getsource(HerReplayBuffer._sample_goal))

ModuleNotFoundError: No module named 'gymnasium'

## Implementacja — automatyczne $\alpha$ w SAC

Pełny moduł: [`asdf/algos.py`](asdf/algos.py). Współczynnik temperatury entropii jest uczony jako `log_alpha` (optymalizacja Adam), zgodnie z sekcją *Automating Entropy Adjustment* w [SAC Algorithms and Applications](https://arxiv.org/abs/1812.05905). Poniżej: fragment `__init__` (inicjalizacja), `compute_loss_alpha` oraz krok w `update`.

In [ ]:
import inspect

from asdf.algos import SAC

src_init = inspect.getsource(SAC.__init__)
needle = 'if isinstance(alpha, str) and alpha.startswith("auto"):'
i = src_init.find(needle)
if i < 0:
    raise RuntimeError("Nie znaleziono bloku auto-alpha w SAC.__init__")
print("--- SAC.__init__ (auto α) ---\n")
print(src_init[i:])

print("\n--- SAC.compute_loss_alpha ---\n")
print(inspect.getsource(SAC.compute_loss_alpha))

print("\n--- SAC.update (fragment: optymalizacja α) ---\n")
src_upd = inspect.getsource(SAC.update)
j = src_upd.find("# Optimize entropy coefficient")
print(src_upd[j:] if j >= 0 else src_upd)

## Wyniki — krzywe z TensorBoard (`runs/`)

Wybierany jest **najnowszy** podkatalog w `runs/` (po czasie modyfikacji plików wewnątrz). Jeśli chcesz konkretny run, zmień zmienną `TB_RUN` w komórce poniżej (ścieżka do podkatalogu z `events.out.tfevents.*`).

In [ ]:
from tensorboard.backend.event_processing import event_accumulator as ea


def pick_latest_tensorboard_run(runs_dir: Path) -> Path:
    if not runs_dir.is_dir():
        raise FileNotFoundError(f"Brak katalogu TensorBoard: {runs_dir}")
    subs = [p for p in runs_dir.iterdir() if p.is_dir()]
    if not subs:
        raise FileNotFoundError(f"Brak podkatalogów w {runs_dir}")

    def latest_mtime(d: Path) -> float:
        files = list(d.glob("*"))
        if not files:
            return d.stat().st_mtime
        return max(f.stat().st_mtime for f in files)

    return max(subs, key=latest_mtime)


# Ustaw na None aby wybrać najnowszy; albo np. RUNS_DIR / "May24_12-00-00"
TB_RUN: Path | None = None
tb_run = TB_RUN or pick_latest_tensorboard_run(RUNS_DIR)
print("TensorBoard:", tb_run)

acc = ea.EventAccumulator(str(tb_run), size_guidance={"scalars": 0})
acc.Reload()
scalar_tags = acc.Tags().get("scalars", [])
print("Skalary:", sorted(scalar_tags))


def plot_scalar(tag: str, title: str | None = None) -> None:
    if tag not in scalar_tags:
        print(f"(brak tagu {tag!r})")
        return
    events = acc.Scalars(tag)
    steps = [e.step for e in events]
    vals = [e.value for e in events]
    plt.figure(figsize=(9, 3.2))
    plt.plot(steps, vals, lw=1)
    plt.xlabel("krok środowiska")
    plt.ylabel(tag)
    plt.title(title or tag)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


plt.rcParams["figure.dpi"] = 110
plot_scalar("test_ep_return", "Test: średni zwrot (deterministyczna polityka)")
plot_scalar("test_ep_length", "Test: średnia długość epizodu")
plot_scalar("alpha", "Temperatura entropii α (uczona)")
plot_scalar("loss_q")
plot_scalar("loss_pi")
plot_scalar("loss_alpha", "Strata dla log α")
plot_scalar("ep_return", "Trening: zwrot ostatniego epizodu (log)")
plot_scalar("ep_length", "Trening: długość ostatniego epizodu")

## Wyniki — ewaluacja z checkpointu

Wczytanie `policy.pt` po weryfikacji `signature` względem `train.py` (`load_sac_from_run_dir`). Krótki test deterministyczny na tym samym `env_id` co trening.

In [ ]:
import gymnasium as gym
import panda_gym  # noqa: F401 — rejestracja środowisk

from train import load_sac_from_run_dir

env, algo, doc = load_sac_from_run_dir(RUN_DIR)
n_ep = 50
results = algo.test(env, n_episodes=n_ep, render=False)
env.close()

print(f"Epizodów testowych: {n_ep}")
for k, v in results.items():
    if isinstance(v, (float, np.floating, np.integer, int)):
        print(f"  {k}: {float(v):.4f}")

print("\n--- Skrót do sekcji „Wyniki liczbowe” (markdown powyżej) ---")
_line = f"Średni zwrot testowy: {float(results['mean_ep_ret']):.4f}; średnia długość: {float(results['mean_ep_len']):.2f}"
if "success_rate" in results:
    _line += f"; success rate: {float(results['success_rate']):.4f}"
print(_line)


def eval_episode_returns(env_id: str, algo, n_episodes: int, seed: int = 0) -> tuple[list[float], list[int]]:
    rng = np.random.default_rng(seed)
    env_e = gym.make(env_id)
    rets, lens = [], []
    try:
        for _ in range(n_episodes):
            obs, _ = env_e.reset(seed=int(rng.integers(0, 2**31 - 1)))
            done, ep_ret, ep_len = False, 0.0, 0
            while not done and ep_len < algo.max_episode_len:
                a = algo.policy.act(obs, deterministic=True)
                obs, r, ter, tru, _ = env_e.step(a)
                ep_ret += float(r)
                ep_len += 1
                done = ter or tru
            rets.append(ep_ret)
            lens.append(ep_len)
    finally:
        env_e.close()
    return rets, lens


env_id = doc["signature"]["env_id"]
rets, lens = eval_episode_returns(env_id, algo, n_episodes=100)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].hist(rets, bins=20, color="steelblue", edgecolor="white")
ax[0].set_title("Rozkład zwrotów (100 ep., deterministycznie)")
ax[0].set_xlabel("zwrot")
ax[1].hist(lens, bins=20, color="coral", edgecolor="white")
ax[1].set_title("Rozkład długości epizodów")
ax[1].set_xlabel("kroki")
plt.tight_layout()
plt.show()

## Analiza i wnioski

### Środowisko i ustawienia HER

- Zadanie **PandaReach-v3** ma nagrodę opartą o odległość do `desired_goal`; bez HER wiele trajektorii daje bardzo mało sygnału uczącego.
- W eksperymencie użyto strategii **`future`** oraz **`n_sampled_goal` = 3** (patrz `signature` w komórce z `metadata.json`): dla każdego kroku epizodu, oprócz zapisu z prawdziwym celem, do bufora trafiają dodatkowe przejścia z **przekształconym** `desired_goal` (losowy `achieved_goal` z „przyszłości” epizodu) i **przeliczoną** nagrodą z `compute_reward`. To zwiększa gęstość przykładów dla sieci Q i polityki.

### Automatyczne $\alpha$

- Krzywa **`alpha`** (TensorBoard) pokazuje, jak temperatura entropii dostosowuje się do skali log-prawdopodobieństwa akcji względem zadanego `target_entropy` (tu: „auto”, czyli $-\dim \mathcal{A}$).
- **`loss_alpha`** oscyluje wokół zera, gdy średnie $\log \pi$ zbliża się do $-H_\text{target}$ — to oczekiwane przy poprawnie zestrojonym automacie.

### Wyniki liczbowe

- Uruchom komórkę z `algo.test` — na końcu wypisze się **jedna linia „Skrót…”**; wklej ją tutaj zamiast tej ramki:

> *(tu wklej linię z outputu, np. średni zwrot, długość epizodu, success rate)*

- Porównaj z krzywą **`test_ep_return`** z TensorBoard: czy widać wzrost / plateau / regresję pod koniec treningu?

### Wyzwania implementacyjne

- **Bufor episodyczny:** przejścia są składowane do `end_episode`; na końcu `train()` jest flush ostatniego epizodu — bez tego część danych HER nie trafiłaby do bufora.
- **Zgodność metadanych:** `train.py --load` wymaga identycznej sygnatury z `metadata.json` — zmiana hiperparametrów w kodzie unieważnia stare wagi.
- **Render:** pełny pipeline z `render_mode="human"` wymaga display / `xvfb-run` (opis w `README.md` w Dockerze).

### Dalsza praca (opcjonalnie)

- Inne środowiska z komentarza w `train.py` (np. `FetchReach-v3`, `PandaPush-v3`) — inna dynamika; HER bywa tam szczególnie przydatny.

---
*Sprawozdanie — sekcję wyników liczbowych uzupełnij linią z outputu po uruchomieniu notebooka od góry do dołu.*